# SWE-bench Verified on A100 (Docker) — HiveMind code_duo

This notebook runs the **official SWE-bench Verified** tasks through the real
HiveMind `code_duo` coder. The official SWE-bench evaluator remains the judge.

**Requirements**
- Linux GPU VM/notebook with **Docker** (Colab A100 does **not** provide a usable
  Docker daemon; use a Docker-capable Jupyter such as RunPod/Lightning/AWS),
  root or docker group, ~120 GB free disk for the full set.
- A100 80 GB recommended (Qwen3.6-35B-A3B UD-Q4_K_XL, full GPU offload).
- Start with 5 tasks; a full 500-task run is many GPU-hours.

The harness mounts each official task image's `/testbed` into a host workspace,
routes HiveMind's `run_bash` through `docker exec`, captures the model patch and
writes `predictions.jsonl` for the official evaluator.


In [ ]:
# 1) GPU + Docker + disk
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!docker info >/dev/null 2>&1 && echo DOCKER_OK || (echo DOCKER_MISSING && false)
!df -h /root | tail -1


In [ ]:
# 2) Clone/update dev + bootstrap
![ -d /root/HiveMind_llama/.git ] || git clone --branch dev https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git fetch origin dev && git reset --hard origin/dev
!cp /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='%h %s'
# Optional: if the harness branch is not on dev yet, upload
# swebench_harness_bundle.tgz to /root and this overlays it.
!test -f /root/swebench_harness_bundle.tgz   && tar xzf /root/swebench_harness_bundle.tgz -C /root/HiveMind_llama   || true
# This notebook needs the benchmark container hook in the engine.
!test -f /root/HiveMind_llama/bench/external/swebench_hivemind.py \
  && grep -q bench_container /root/HiveMind_llama/tools/handlers/exec_tools.py \
  && echo HARNESS_OK || (echo HARNESS_MISSING && false)


In [ ]:
# 3) Setup (~15-30 min: Python 3.14, llama.cpp CUDA 12.8, 20 GB model)
!HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup


In [ ]:
# 4) Start HiveMind + load the recommended 35B on the A100
!grep -q 'vram_budget_gb' /root/rent_bootstrap.sh && echo BOOTSTRAP_OK || echo BOOTSTRAP_STALE
!bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health


In [ ]:
# 5) Separate Python 3.12 env for the official SWE-bench package
!command -v /root/.local/bin/uv >/dev/null 2>&1 || curl -LsSf https://astral.sh/uv/install.sh | sh
!export PATH=/root/.local/bin:$PATH && uv venv --python 3.12 /root/swebench-venv
!export PATH=/root/.local/bin:$PATH && uv pip install --python /root/swebench-venv/bin/python -q httpx datasets swebench
!/root/swebench-venv/bin/python -c "import swebench, datasets, httpx; print('bench env OK')" 


In [ ]:
# 6) Export SWE-bench Verified to local JSONL (500 rows)
%%bash
/root/swebench-venv/bin/python - <<'PY'
import json
from datasets import load_dataset

ds = load_dataset("SWE-bench/SWE-bench_Verified", split="test")
out = "/root/swebench_verified.jsonl"
with open(out, "w", encoding="utf-8") as fh:
    for row in ds:
        fh.write(json.dumps({
            "instance_id": row["instance_id"],
            "repo": row["repo"],
            "base_commit": row["base_commit"],
            "problem_statement": row["problem_statement"],
        }) + "\n")
print("wrote", out, len(ds), "instances")
PY


In [ ]:
# 7) 5-task spike: HiveMind code_duo + official task containers
%%bash
set -euo pipefail
cd /root/HiveMind_llama
OUT="/root/HiveMind_llama/bench/results/swebench-hivemind-$(date +%Y%m%d-%H%M%S)"
/root/swebench-venv/bin/python -u bench/external/swebench_hivemind.py \
  --dataset /root/swebench_verified.jsonl \
  --n-tasks 5 \
  --hivemind http://127.0.0.1:8001 \
  --model qwen3.6:35b-a3b-ud \
  --ctx 32768 --tool-rounds 50 \
  --out "$OUT"
echo "OUT=$OUT" 


In [ ]:
# 8) Official SWE-bench evaluation of the generated predictions
%%bash
set -euo pipefail
cd /root/HiveMind_llama
OUT=$(ls -td bench/results/swebench-hivemind-* | head -1)
RUN_ID="hivemind-$(date +%Y%m%d-%H%M%S)"
/root/swebench-venv/bin/python -m swebench.harness.run_evaluation \
  --dataset_name SWE-bench/SWE-bench_Verified \
  --predictions_path "$OUT/predictions.jsonl" \
  --run_id "$RUN_ID" \
  --max_workers 4
echo "RUN_ID=$RUN_ID"
echo "RESULTS=logs/evaluation/$RUN_ID/results.json" 


In [ ]:
# 9) Save harness output + official eval logs
%%bash
set -euo pipefail
cd /root/HiveMind_llama
OUT=$(ls -td bench/results/swebench-hivemind-* | head -1)
mkdir -p /root/swebench_out
cp -rv "$OUT" /root/swebench_out/
cp -rv logs/evaluation /root/swebench_out/ 2>/dev/null || true
tar czf /root/swebench_out.tgz -C /root swebench_out
ls -lh /root/swebench_out.tgz


In [ ]:
# 10) Download the bundle (Colab) or use the Jupyter file browser
try:
    from google.colab import files
    files.download('/root/swebench_out.tgz')
except Exception:
    print("Download /root/swebench_out.tgz from the Jupyter file browser")


## Full run / shards

For a full SWE-bench Verified run, do **not** use `--n-tasks 5`:

```bash
--n-tasks 0
```

To split across four workers/engines, run four shards with the same dataset and
different output dirs:

```bash
--n-tasks 0 --shard-index 0 --shards 4
--n-tasks 0 --shard-index 1 --shards 4
--n-tasks 0 --shard-index 2 --shards 4
--n-tasks 0 --shard-index 3 --shards 4
```

Merge the prediction files and evaluate them together:

```bash
cat bench/results/swebench-hivemind-*/predictions.jsonl > /root/all_predictions.jsonl
```

The official evaluator caches by `run_id`; use a new run id for every new set of
predictions. Three runs are required for a publishable pass@1 with confidence
intervals.
